# Solutions · 00 · Python for reaction engineering

Worked solutions to the exercises of [notebook 00](../notebooks/00_python_for_reaction_engineering.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate, optimize
from engreact import kinetics
from engreact.constants import R
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A reaction has Ea = 100 kJ/mol. By what factor does the rate rise between 25 and 35 degC, and between 300 and 310 degC? Why is the 'rate doubles per 10 degrees' rule only a rule of thumb?

In [2]:
Ea = 100e3
for T1, T2 in ((298.15, 308.15), (573.15, 583.15)):
    print(f"{T1-273.15:.0f} -> {T2-273.15:.0f} degC: rate x {kinetics.rate_ratio(Ea, T1, T2):.2f}")
print("the 'doubling per 10 K' rule holds for Ea ~ 50 kJ/mol near room temperature only: the factor is exp(Ea/R x 10/T^2), which falls as T rises")

25 -> 35 degC: rate x 3.70
300 -> 310 degC: rate x 1.43
the 'doubling per 10 K' rule holds for Ea ~ 50 kJ/mol near room temperature only: the factor is exp(Ea/R x 10/T^2), which falls as T rises


With $E_a$ = 100 kJ/mol the rate rises 3.7-fold per 10 K at room temperature but only 1.4-fold at 300 degC:
the Arrhenius factor for a fixed step depends on $E_a/RT^2$. The rule of thumb encodes one activation
energy at one temperature.

## Exercise 2

Integrate the batch reactor for a second-order reaction (-r = k C^2, k = 0.11 L/(mol s), C0 = 0.02 mol/L) with solve_ivp and compare with the closed form C = C0/(1 + k C0 t) at 5, 10 and 20 min.

In [3]:
k2, C0 = 0.11, 0.02                                          # L/(mol s), mol/L
sol = integrate.solve_ivp(lambda t, C: [-k2 * C[0] ** 2], (0, 1200), [C0], t_eval=[300, 600, 1200], rtol=1e-10, atol=1e-14)
exact = C0 / (1 + k2 * C0 * sol.t)
print(pd.DataFrame({"t (min)": sol.t / 60, "solve_ivp": sol.y[0], "closed form": exact, "rel. error": sol.y[0] / exact - 1}).to_string(index=False, float_format=lambda v: f"{v:.6g}"))

 t (min)  solve_ivp  closed form  rel. error
       5  0.0120482    0.0120482 6.33271e-13
      10 0.00862069   0.00862069 6.14375e-12
      20 0.00549451   0.00549451 3.27505e-11


The integrator reproduces the closed form to eight digits with tight tolerances - and the closed form is
what tells you the tolerances were tight enough. Always keep an analytic case in reach.

## Exercise 3

**Implement it yourself:** write a function that returns the temperature at which a rate constant reaches a given value (invert Arrhenius) and check it against `kinetics.temperature_for_rate_ratio`.

In [4]:
def temperature_for_k(k0, Ea, k_target):
    return -Ea / (R * np.log(k_target / k0))
k_target = kinetics.arrhenius(5.0e7, 68e3, 300.0) * 10
T_hand = temperature_for_k(5.0e7, 68e3, k_target)
print(f"by hand: {T_hand:.2f} K; library: {kinetics.temperature_for_rate_ratio(68e3, 300.0, 10.0):.2f} K")

by hand: 327.68 K; library: 327.68 K


Inverting $k = k_0 e^{-E_a/RT}$ is one logarithm; both routes give the temperature for a tenfold rate.